# Data Cleaning & Transformation

This notebook transforms the raw DataCo Smart Supply Chain dataset into a clean, analysis-ready dataset.

The raw source data is preserved unchanged in `data/raw/`.

## Objectives

1. Load the raw dataset
2. Remove unnecessary and sensitive fields
3. Handle missing values
4. Convert columns to appropriate data types
5. Remove redundant or non-informative variables
6. Standardize column names and categorical values where necessary
7. Create useful analytical features
8. Validate the cleaned dataset
9. Export the processed dataset for downstream analysis

## Cleaning Principles

- The raw dataset will never be modified.
- Every transformation must have a documented analytical reason.
- Missing values will not be filled without justification.
- Potential outliers will not be removed automatically.
- Identifier columns will be distinguished from numerical measures.
- Sensitive or unnecessary personal information will not be retained in the analytical dataset.
- All transformations should be reproducible.

In [29]:
import pandas as pd
import numpy as np

In [30]:
# Load raw dataset

file_path = "../data/raw/DataCoSupplyChainDataset.csv"

df = pd.read_csv(
    file_path,
    encoding="latin-1"
)

print("Raw dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

Raw dataset loaded successfully.
Rows: 180,519
Columns: 53


## Cleaning Decision Framework

The raw dataset contains 53 columns. Each column will be evaluated based on:

- Analytical usefulness
- Data completeness
- Privacy considerations
- Redundancy
- Variability
- Suitability for downstream analysis

Columns will be classified as:

- **RETAIN** — useful for analysis
- **TRANSFORM** — useful but requires modification
- **REMOVE** — unnecessary, redundant, sensitive, or non-informative
- **INVESTIGATE** — requires additional validation before a final decision

### Initial Column Decisions

#### Remove

The following columns will be removed from the analytical dataset:

- `Customer Email` — sensitive and unnecessary for business analysis
- `Customer Password` — sensitive and unnecessary
- `Customer Fname` — individual-level identifying information is not required
- `Customer Lname` — individual-level identifying information is not required
- `Customer Street` — unnecessary for the project's analytical objectives
- `Product Description` — 100% missing
- `Product Status` — constant value of 0 across all records

#### Retain

Customer, order, product, category, department, geographic, shipping, and financial identifiers will initially be retained because they support aggregation, validation, relationships, and downstream SQL/BI analysis.

#### Transform

The following fields require transformation:

- `order date (DateOrders)` → datetime
- `shipping date (DateOrders)` → datetime
- Column names → standardized naming convention

#### Investigate

`Order Zipcode` contains approximately 86.24% missing values. It will initially be retained while its analytical usefulness is evaluated.

In [31]:
# Columns identified for removal

columns_to_remove = [
    "Customer Email",
    "Customer Password",
    "Customer Fname",
    "Customer Lname",
    "Customer Street",
    "Product Description",
    "Product Status"
]

df_clean = df.drop(columns=columns_to_remove).copy()

print(f"Original columns: {df.shape[1]}")
print(f"Cleaned columns: {df_clean.shape[1]}")
print(f"Columns removed: {df.shape[1] - df_clean.shape[1]}")

Original columns: 53
Cleaned columns: 46
Columns removed: 7


## Date Transformation

The order and shipping date fields are currently stored as strings.

They will be converted to Pandas datetime objects so they can support:

- Time-based analysis
- Duration calculations
- Monthly and yearly aggregation
- Seasonality analysis
- Demand forecasting

In [32]:
# Convert date columns to datetime

date_columns = [
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]

for column in date_columns:
    df_clean[column] = pd.to_datetime(
        df_clean[column],
        errors="coerce"
    )

print("Date columns converted successfully.\n")
print(df_clean[date_columns].dtypes)

Date columns converted successfully.

order date (DateOrders)       datetime64[us]
shipping date (DateOrders)    datetime64[us]
dtype: object


In [33]:
# Validate date conversion

print("Missing order dates:", df_clean["order date (DateOrders)"].isna().sum())
print("Missing shipping dates:", df_clean["shipping date (DateOrders)"].isna().sum())

Missing order dates: 0
Missing shipping dates: 0


## Investigating Missing Order Zipcodes

`Order Zipcode` contains approximately 86.24% missing values.

Before removing the column, we will investigate whether the available values provide meaningful analytical information and whether equivalent geographic information is already available through other fields.

The dataset already contains:

- Order City
- Order State
- Order Country
- Order Region
- Market

These fields may provide sufficient geographic granularity for the project's objectives.

In [34]:
# Investigate Order Zipcode availability

zipcode_available = df_clean["Order Zipcode"].notna()

print("Orders with Order Zipcode:", zipcode_available.sum())
print("Orders without Order Zipcode:", (~zipcode_available).sum())

print("\nOrder Zipcode availability by Order Country:")
print(
    df_clean.groupby("Order Country")["Order Zipcode"]
    .apply(lambda x: x.notna().mean() * 100)
    .sort_values(ascending=False)
    .head(20)
)

Orders with Order Zipcode: 24840
Orders without Order Zipcode: 155679

Order Zipcode availability by Order Country:
Order Country
Estados Unidos          100.0
Afganistán                0.0
Alemania                  0.0
Albania                   0.0
Arabia Saudí              0.0
Argelia                   0.0
Argentina                 0.0
Armenia                   0.0
Australia                 0.0
Austria                   0.0
Azerbaiyán                0.0
Bangladés                 0.0
Barbados                  0.0
Baréin                    0.0
Belice                    0.0
Benín                     0.0
Bielorrusia               0.0
Bolivia                   0.0
Bosnia y Herzegovina      0.0
Botsuana                  0.0
Name: Order Zipcode, dtype: float64


In [35]:
# Compare geographic information available for orders with and without zipcodes

print("Unique Order Countries:", df_clean["Order Country"].nunique())
print("Unique Order States:", df_clean["Order State"].nunique())
print("Unique Order Cities:", df_clean["Order City"].nunique())
print("Unique Order Regions:", df_clean["Order Region"].nunique())

Unique Order Countries: 164
Unique Order States: 1089
Unique Order Cities: 3597
Unique Order Regions: 23


## Investigating Redundant Variables

Several columns appear to represent the same underlying entity or measure under different names.

Before removing any variables, their values will be compared directly.

A column will only be considered redundant if its values are equivalent across the dataset and another column provides the same analytical information.

In [36]:
# Check potentially redundant identifier columns

redundant_pairs = [
    ("Customer Id", "Order Customer Id"),
    ("Product Card Id", "Order Item Cardprod Id"),
    ("Category Id", "Product Category Id"),
    ("Product Price", "Order Item Product Price")
]

for col1, col2 in redundant_pairs:
    equal = df_clean[col1].equals(df_clean[col2])

    print(f"{col1} == {col2}: {equal}")

Customer Id == Order Customer Id: True
Product Card Id == Order Item Cardprod Id: True
Category Id == Product Category Id: True
Product Price == Order Item Product Price: True


## Validating Shipping Duration

The dataset contains both:

- `Days for shipping (real)`
- Order date
- Shipping date

We will independently calculate the shipping duration from the timestamps and compare it with the provided operational field.

In [37]:
# Calculate actual shipping duration from timestamps

calculated_shipping_days = (
    df_clean["shipping date (DateOrders)"]
    - df_clean["order date (DateOrders)"]
).dt.total_seconds() / (24 * 60 * 60)

difference = (
    calculated_shipping_days
    - df_clean["Days for shipping (real)"]
).abs()

print("Maximum difference:", difference.max())
print("Rows with difference > 0.01 days:", (difference > 0.01).sum())

Maximum difference: 0.5
Rows with difference > 0.01 days: 9737


## Order Zipcode — Cleaning Decision

`Order Zipcode` contains 155,679 missing values out of 180,519 records (86.24%).

Investigation showed that all 24,840 non-null order zipcodes belong to orders from the United States, while international orders generally do not contain zipcode information.

This indicates that the missingness is structural rather than random.

The column will therefore be removed from the primary analytical dataset rather than imputed.

The project retains broader and more consistently available geographic fields:

- `Order Country`
- `Order Region`
- `Order State`
- `Order City`
- `Market`

These provide sufficient geographic granularity for the project's revenue, demand, logistics, and profitability analyses.

In [38]:
# Remove Order Zipcode due to high, structurally missing coverage

df_clean = df_clean.drop(columns=["Order Zipcode"])

print("Order Zipcode removed.")
print("Current number of columns:", df_clean.shape[1])

Order Zipcode removed.
Current number of columns: 45


In [39]:
redundant_pairs = [
    ("Customer Id", "Order Customer Id"),
    ("Product Card Id", "Order Item Cardprod Id"),
    ("Category Id", "Product Category Id"),
    ("Product Price", "Order Item Product Price")
]

for col1, col2 in redundant_pairs:
    equal = df_clean[col1].equals(df_clean[col2])
    print(f"{col1} == {col2}: {equal}")

Customer Id == Order Customer Id: True
Product Card Id == Order Item Cardprod Id: True
Category Id == Product Category Id: True
Product Price == Order Item Product Price: True


## Redundant Column Analysis

Four pairs of columns were found to contain identical values across all 180,519 records:

| Retained Column | Removed Column | Reason |
|---|---|---|
| `Customer Id` | `Order Customer Id` | Identical customer identifier |
| `Product Card Id` | `Order Item Cardprod Id` | Identical product identifier |
| `Category Id` | `Product Category Id` | Identical category identifier |
| `Product Price` | `Order Item Product Price` | Identical product price |

The retained columns use the more general and clearer naming convention.

Removing the duplicates reduces unnecessary data redundancy without losing information.

In [40]:
# Remove redundant duplicate columns

redundant_columns = [
    "Order Customer Id",
    "Order Item Cardprod Id",
    "Product Category Id",
    "Order Item Product Price"
]

df_clean = df_clean.drop(columns=redundant_columns)

print("Redundant columns removed:", len(redundant_columns))
print("Current number of columns:", df_clean.shape[1])

Redundant columns removed: 4
Current number of columns: 41


## Financial Data Validation

The dataset contains several financial variables that are mathematically related.

Before making any changes, we will validate the relationships between these fields to determine whether they contain independent information or represent derived versions of other variables.

The first relationship to validate is:

`Sales - Order Item Discount ≈ Order Item Total`

Small floating-point differences are expected because the source data contains decimal values.

In [41]:
# Validate Sales, Discount, and Order Item Total

calculated_total = (
    df_clean["Sales"]
    - df_clean["Order Item Discount"]
)

difference = (
    calculated_total
    - df_clean["Order Item Total"]
).abs()

print("Maximum absolute difference:", difference.max())
print("Rows with difference > 0.01:", (difference > 0.01).sum())

Maximum absolute difference: 0.010013549999996485
Rows with difference > 0.01: 1224


### Quantity and Sales Validation

We will also verify whether `Sales` is consistent with:

`Product Price × Order Item Quantity`

This helps validate the integrity of the transaction-level financial data.

In [42]:
# Validate Product Price × Quantity = Sales

calculated_sales = (
    df_clean["Product Price"]
    * df_clean["Order Item Quantity"]
)

difference = (
    calculated_sales
    - df_clean["Sales"]
).abs()

print(
    "Maximum absolute difference:",
    difference.max()
)

print(
    "Rows with difference > 0.01:",
    (difference > 0.01).sum()
)

Maximum absolute difference: 2.289999997628911e-05
Rows with difference > 0.01: 0


### Profit Consistency Validation

`Order Item Profit Ratio` appears to represent the profit generated relative to the order item's total value.

We will test whether:

`Order Item Total × Order Item Profit Ratio ≈ Order Profit Per Order`

This will help determine whether the profit fields are internally consistent.

In [43]:
# Validate profit calculation

calculated_profit = (
    df_clean["Order Item Total"]
    * df_clean["Order Item Profit Ratio"]
)

difference = (
    calculated_profit
    - df_clean["Order Profit Per Order"]
).abs()

print("Maximum absolute difference:", difference.max())
print("Rows with difference > 0.01:", (difference > 0.01).sum())

Maximum absolute difference: 9.299195480020018
Rows with difference > 0.01: 65390


In [44]:
# Compare profit-related columns

difference = (
    df_clean["Benefit per order"]
    - df_clean["Order Profit Per Order"]
).abs()

print("Maximum absolute difference:", difference.max())
print("Rows with difference > 0.01:", (difference > 0.01).sum())

Maximum absolute difference: 0.0
Rows with difference > 0.01: 0


In [45]:
# Compare profit ratio against actual profit and sales

calculated_ratio = (
    df_clean["Order Profit Per Order"]
    / df_clean["Sales"]
)

difference = (
    calculated_ratio
    - df_clean["Order Item Profit Ratio"]
).abs()

print("Maximum absolute ratio difference:", difference.max())
print("Rows with difference > 0.01:", (difference > 0.01).sum())

Maximum absolute ratio difference: 0.6874968765626641
Rows with difference > 0.01: 127905


In [46]:
# Inspect records with large profit-calculation discrepancies

profit_check = df_clean[
    [
        "Sales",
        "Order Item Total",
        "Order Item Profit Ratio",
        "Order Profit Per Order",
        "Benefit per order"
    ]
].copy()

profit_check["Calculated Profit"] = (
    profit_check["Order Item Total"]
    * profit_check["Order Item Profit Ratio"]
)

profit_check["Difference"] = (
    profit_check["Calculated Profit"]
    - profit_check["Order Profit Per Order"]
).abs()

profit_check.sort_values(
    "Difference",
    ascending=False
).head(10)

,Sales,Order Item Total,Order Item Profit Ratio,Order Profit Per Order,Benefit per order,Calculated Profit,Difference
173436,1999.98999,1859.98999,0.08,139.500000,139.500000,148.799195,9.299195
13343,1999.98999,1759.98999,0.38,660.000000,660.000000,668.796187,8.796187
59847,1999.98999,1599.98999,0.33,520.000000,520.000000,527.996717,7.996717
38434,1500.00000,1500.00000,-0.23,-337.500000,-337.500000,-345.000006,7.500006
26354,1500.00000,1500.00000,0.08,112.500000,112.500000,119.999997,7.499997
50277,1500.00000,1485.00000,-0.73,-1076.630005,-1076.630005,-1084.050028,7.420023
127420,1500.00000,1485.00000,0.38,556.880005,556.880005,564.299993,7.419988
15852,1500.00000,1470.00000,0.33,477.750000,477.750000,485.100019,7.350019
160159,1500.00000,1470.00000,-0.23,-330.750000,-330.750000,-338.100006,7.350006
120687,1500.00000,1470.00000,-0.88,-1286.250000,-1286.250000,-1293.599993,7.349993


In [47]:
# Calculate the profit ratio implied by the recorded profit

actual_profit_ratio = (
    df_clean["Order Profit Per Order"]
    / df_clean["Sales"]
)

ratio_difference = (
    actual_profit_ratio
    - df_clean["Order Item Profit Ratio"]
).abs()

print("Maximum absolute ratio difference:",
      ratio_difference.max())

print("Rows with difference > 0.01:",
      (ratio_difference > 0.01).sum())

print("\nSample comparison:")

ratio_check = df_clean[
    [
        "Sales",
        "Order Profit Per Order",
        "Order Item Profit Ratio"
    ]
].copy()

ratio_check["Calculated Profit Ratio"] = (
    ratio_check["Order Profit Per Order"]
    / ratio_check["Sales"]
)

ratio_check.head(10)

Maximum absolute ratio difference: 0.6874968765626641
Rows with difference > 0.01: 127905

Sample comparison:


,Sales,Order Profit Per Order,Order Item Profit Ratio,Calculated Profit Ratio
0,327.75,91.250000,0.29,0.278413
1,327.75,-249.089996,-0.80,-0.760000
2,327.75,-247.779999,-0.80,-0.756003
3,327.75,22.860001,0.08,0.069748
4,327.75,134.210007,0.45,0.409489
5,327.75,18.580000,0.06,0.056690
6,327.75,95.180000,0.33,0.290404
7,327.75,68.430000,0.24,0.208787
8,327.75,133.720001,0.48,0.407994
9,327.75,132.149994,0.48,0.403204


## Removing Redundant Profit Column

The dataset contains both `Benefit per order` and `Order Profit Per Order`.

A direct comparison showed that the two columns are exactly identical:

- Maximum absolute difference: `0.0`
- Rows with difference > `0.01`: `0`

Since both columns contain the same information, retaining both would introduce unnecessary redundancy.

`Order Profit Per Order` is retained because its name more explicitly describes the metric used in this project.

Therefore, `Benefit per order` will be removed from the cleaned dataset.

In [48]:
# Remove redundant profit column

df_clean = df_clean.drop(columns=["Benefit per order"])

print("Current number of columns:", df_clean.shape[1])

Current number of columns: 40


In [49]:
print(df_clean.columns.tolist())

['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Id', 'Customer Segment', 'Customer State', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'order date (DateOrders)', 'Order Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Product Card Id', 'Product Image', 'Product Name', 'Product Price', 'shipping date (DateOrders)', 'Shipping Mode']


## Remaining Missing Values

After removing irrelevant, redundant, and structurally incomplete columns, the cleaned dataset is checked again for remaining missing values.

The objective is to determine whether any remaining missing values require:

- removal of a column,
- imputation,
- or no action because the missingness is negligible and does not affect the planned analysis.

No values will be imputed without first understanding why they are missing.

In [50]:
# Re-check missing values after structural cleaning

remaining_missing = df_clean.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0].sort_values(ascending=False)

print("Remaining columns with missing values:")
print(remaining_missing)

print("\nTotal missing values:", df_clean.isnull().sum().sum())

Remaining columns with missing values:
Customer Zipcode    3
dtype: int64

Total missing values: 3


## Investigating Remaining Missing Customer Zipcodes

After structural cleaning, only three missing values remain in the dataset, all in `Customer Zipcode`.

Because the number of missing values is extremely small (3 out of 180,519 rows), these records will not justify removing the entire column.

Before deciding whether to impute or retain the missing values, the corresponding customer location information will be inspected to determine whether the missing zipcodes can be reliably inferred.

In [51]:
# Inspect the records with missing Customer Zipcode

missing_customer_zipcode = df_clean[
    df_clean["Customer Zipcode"].isna()
][
    [
        "Customer Id",
        "Customer City",
        "Customer State",
        "Customer Country",
        "Customer Zipcode"
    ]
]

missing_customer_zipcode

,Customer Id,Customer City,Customer State,Customer Country,Customer Zipcode
35704,14577,CA,95758,EE. UU.,NaN
46440,17171,CA,95758,EE. UU.,NaN
82511,14046,CA,91732,EE. UU.,NaN


## Handling Remaining Missing Customer Zipcodes

The investigation identified only three records with missing `Customer Zipcode` values.

These records represent an extremely small proportion of the dataset (3 out of 180,519 rows, approximately 0.0017%).

The available customer location fields do not provide sufficient information to reliably reconstruct the missing ZIP codes. Therefore, the values will be retained as missing rather than being artificially imputed.

The `Customer Zipcode` column will be retained because it may still be useful for geographic analysis where the value is available.

No rows will be removed because of these three missing values.

In [52]:
# Verify the final missing-value status

remaining_missing = df_clean.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

print("Remaining missing values:")
print(remaining_missing)

print("\nTotal missing values:", df_clean.isnull().sum().sum())

Remaining missing values:
Customer Zipcode    3
dtype: int64

Total missing values: 3


## Data Type Normalization

The raw dataset contains several columns whose data types require normalization before analysis.

In particular, the order and shipping date columns were originally stored as strings. These will be converted to proper datetime types so that the dataset can support:

- time-based aggregation
- trend analysis
- seasonality analysis
- delivery-duration calculations
- demand forecasting

Numeric columns will also be checked to ensure that identifiers, quantities, financial measures, and categorical flags have appropriate representations.

The original raw dataset remains unchanged; all transformations are applied to `df_clean`.

In [53]:
# Inspect current data types

print(df_clean.dtypes)

Type                                        str
Days for shipping (real)                  int64
Days for shipment (scheduled)             int64
Sales per customer                      float64
Delivery Status                             str
Late_delivery_risk                        int64
Category Id                               int64
Category Name                               str
Customer City                               str
Customer Country                            str
Customer Id                               int64
Customer Segment                            str
Customer State                              str
Customer Zipcode                        float64
Department Id                             int64
Department Name                             str
Latitude                                float64
Longitude                               float64
Market                                      str
Order City                                  str
Order Country                           

## Data Type Validation

The cleaned dataset was inspected after the data-type transformations.

The order and shipping date columns are now stored as proper datetime values, while financial measures, quantities, identifiers, and categorical fields have appropriate data types for analysis.

`Customer Zipcode` remains a `float64` column because it contains three missing values. This is expected behavior and does not require conversion at this stage.

No additional data-type transformations are required before proceeding to feature engineering and exploratory analysis.

In [54]:
# Final validation of date columns

date_columns = [
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]

print("Date column validation:")
for column in date_columns:
    print(f"\n{column}")
    print("Data type:", df_clean[column].dtype)
    print("Missing values:", df_clean[column].isna().sum())
    print("Unique values:", df_clean[column].nunique())

Date column validation:

order date (DateOrders)
Data type: datetime64[us]
Missing values: 0
Unique values: 65752

shipping date (DateOrders)
Data type: datetime64[us]
Missing values: 0
Unique values: 63701


## Final Data Quality Audit

Before saving the cleaned dataset, a final quality audit is performed to verify:

- dataset dimensions
- duplicate records
- missing values
- duplicate identifiers
- data types
- date validity
- numerical consistency

This audit provides a final checkpoint between data cleaning and exploratory analysis.

The cleaned dataset will only be saved after these checks confirm that no unresolved structural issues remain.

In [55]:
# Final data quality audit

print("=" * 60)
print("FINAL DATA QUALITY AUDIT")
print("=" * 60)

# Dataset dimensions
print("\nDataset dimensions:")
print("Rows:", df_clean.shape[0])
print("Columns:", df_clean.shape[1])

# Duplicate rows
print("\nDuplicate rows:", df_clean.duplicated().sum())

# Missing values
missing_values = df_clean.isnull().sum()
missing_values = missing_values[missing_values > 0]

print("\nRemaining missing values:")
print(missing_values if len(missing_values) > 0 else "None")

# Duplicate Order Item IDs
print("\nDuplicate Order Item IDs:",
      df_clean["Order Item Id"].duplicated().sum())

# Date validity
print("\nInvalid date records:")

order_dates = df_clean["order date (DateOrders)"]
shipping_dates = df_clean["shipping date (DateOrders)"]

print("Missing order dates:", order_dates.isna().sum())
print("Missing shipping dates:", shipping_dates.isna().sum())
print("Shipping before order:",
      (shipping_dates < order_dates).sum())

# Numeric consistency checks

sales_check = (
    df_clean["Sales"]
    - (
        df_clean["Product Price"]
        * df_clean["Order Item Quantity"]
    )
).abs()

print("\nSales consistency:")
print("Maximum difference:", sales_check.max())
print("Rows with difference > 0.01:",
      (sales_check > 0.01).sum())

# Final data types
print("\nData types:")
print(df_clean.dtypes.value_counts())

print("\n" + "=" * 60)
print("AUDIT COMPLETE")
print("=" * 60)

FINAL DATA QUALITY AUDIT

Dataset dimensions:
Rows: 180519
Columns: 40

Duplicate rows: 0

Remaining missing values:
Customer Zipcode    3
dtype: int64

Duplicate Order Item IDs: 0

Invalid date records:
Missing order dates: 0
Missing shipping dates: 0
Shipping before order: 0

Sales consistency:
Maximum difference: 2.289999997628911e-05
Rows with difference > 0.01: 0

Data types:
str               17
float64           11
int64             10
datetime64[us]     2
Name: count, dtype: int64

AUDIT COMPLETE


## Saving the Cleaned Dataset

The final data-quality audit confirms that the cleaned dataset is structurally consistent and ready for downstream analysis.

The cleaned dataset contains 180,519 order-item records and 40 columns. No duplicate records or duplicate order-item identifiers remain, both date fields are valid, and the core sales calculation is consistent within the defined numerical tolerance.

The three remaining missing `Customer Zipcode` values are retained because they represent an insignificant proportion of the dataset and cannot be reliably inferred.

The validated dataframe will now be exported to `data/processed/` for use in subsequent analytical stages.

In [28]:
# Save the validated cleaned dataset

import os

processed_dir = "../data/processed"
os.makedirs(processed_dir, exist_ok=True)

output_path = os.path.join(
    processed_dir,
    "supply_chain_cleaned.csv"
)

df_clean.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully.")
print("Path:", output_path)
print("Shape:", df_clean.shape)

Cleaned dataset saved successfully.
Path: ../data/processed\supply_chain_cleaned.csv
Shape: (180519, 40)
